Inspired by [python-agent-framework-msfoundry-file-search.ipynb](https://github.com/microsoft/Agent-Framework-Samples/blob/main/00.ForBeginners/05-agentic-rag/code_samples/python-agent-framework-msfoundry-file-search.ipynb)

# Common Libraries and Global Variables

In [1]:
# Common Libraries
import os, sys
from dotenv import load_dotenv # requires python-dotenv
from IPython.display import Markdown, display

config_path = "../../../config" # explicit path to the config folder
sys.path.append(config_path)
from auth import acquire_bearer_token, StaticBearerTokenCredential
if not load_dotenv(f"{config_path}/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

# Global libraries - recall to declare them as "global" in the functions where they are assigned
project_endpoint = "" # must be Foundry V1 project!
agent_name = ""
deployment_name = ""
bearer_token_cognitiveservices = ""
user_cognitiveservices = ""

Environment variables have been loaded ;-)


# Authentication & Environment setup

In [2]:
def authentication_and_environmentsetup():
    global bearer_token_cognitiveservices, bearer_token_azureai
    bearer_token_cognitiveservices, user_cognitiveservices = acquire_bearer_token(
        scope="https://cognitiveservices.azure.com/.default", # https://ai.azure.com/.default, https://cognitiveservices.azure.com/.default...
        auth_method="default") # default, cli, device

    bearer_token_azureai, _ = acquire_bearer_token(
        scope="https://ai.azure.com/.default", # https://ai.azure.com/.default, https://cognitiveservices.azure.com/.default...
        auth_method="default") # default, cli, device

    print("Bearer token Cognitive Services:", bearer_token_cognitiveservices[:10], "...")
    print("Bearer token Azure AI:", bearer_token_azureai[:10], "...")
    print(f'User info Cognitive Services: {user_cognitiveservices["name"]}, upn: {user_cognitiveservices["upn"]}')
    # bearer_token_cognitiveservices['raw_claims']

authentication_and_environmentsetup()

Bearer token Cognitive Services: eyJ0eXAiOi ...
Bearer token Azure AI: eyJ0eXAiOi ...
User info Cognitive Services: Mauro Minella, upn: mauro.minella@MngEnvMCAP883652.onmicrosoft.com


# Initialization

In [3]:
def init():
    global project_endpoint, deployment_name, agent_name
    
    project_endpoint = os.getenv("AIF_BAS_PROJECT_ENDPOINT") # must be Foundry V1 project
    deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
    agent_name="PythonRAGAgent-sync"
    
init()

# Common elements

In [4]:
# Define the query for the agent

query = "Can you explain Contoso's travel insurance coverage?"
file_path = './document.md'
agent_instructions="""
You are an AI assistant designed to answer user questions using only the retrieved from the provided document(s).
If a user's question cannot be answered using the retrieved context, you must clearly respond:
'I'm sorry, but the uploaded document does not contain the necessary information to answer that question.'
Do not answer from general knowledge or reasoning. Do not make assumptions or generate hypothetical explanations
For questions that do have relevant content in the document, respond accurately and cite the document explicitly.
"""

credential = StaticBearerTokenCredential(bearer_token_azureai)

# Build and Run the Agent Synchronously in multiple cells

In [5]:
from azure.ai.agents import AgentsClient as AgentsClient_sync

agents_client_sync = AgentsClient_sync (endpoint=project_endpoint, credential=credential)

In [6]:
# here we delete all files stored on the service

for f in agents_client_sync.files.list()["data"]:
    print(f"deleting file {f.filename} ({f.id})...")
    agents_client_sync.files.delete(file_id = f.id)

deleting file document.md (assistant-CLue8nD6CzRoRCSUg3TQ9B)...


In [7]:
# 1. Upload file and create vector store

print(f"Uploading file from: {file_path}")
file = agents_client_sync.files.upload_and_poll(file_path=file_path, purpose="assistants")
print(f"Uploaded file, file ID: {file.id}")

Uploading file from: ./document.md
Uploaded file, file ID: assistant-XPJgh79NK9ViHMViChEgBo


In [8]:
# 2. Create file search tool with uploaded resources

from agent_framework.azure import AzureOpenAIResponsesClient

vector_store = agents_client_sync.vector_stores.create_and_poll(file_ids=[file.id], name="contoso_knowledge_base")
file_search_tool = AzureOpenAIResponsesClient.get_file_search_tool(vector_store_ids=[vector_store.id])
print(f"Created vector store, vector store ID: {vector_store.id}")

Created vector store, vector store ID: vs_jDe9djVDrEMIkFOR7Vw5Jvji


In [9]:
# 3. Create an agent with file search capabilities using the provider

from agent_framework.azure import AzureOpenAIResponsesClient

client = AzureOpenAIResponsesClient(
    deployment_name=deployment_name,
    project_endpoint=project_endpoint,
    credential=credential)

agent = client.as_agent(
    name=agent_name,
    instructions=agent_instructions,
    tools=[file_search_tool])

In [10]:
# 4. Query the agent
print(f"\n# User: '{query}'")
response = await agent.run(query)
print(f"# Agent: {response.text}")


# User: 'Can you explain Contoso's travel insurance coverage?'
# Agent: Contoso's travel insurance covers medical emergencies, trip cancellations, and lost baggage.


# Build and Run the Agent Asynchronously in a single cell

In [11]:
from azure.ai.agents.aio import AgentsClient as AgentClient_async
from agent_framework.azure import AzureOpenAIResponsesClient

file = None
vector_store = None

async with (
    AgentClient_async(endpoint=project_endpoint, credential=credential) as agents_client_async,
):
    try:
        client = AzureOpenAIResponsesClient(
            deployment_name=deployment_name,
            project_endpoint=project_endpoint,
            credential=credential)
        
        # 1. Upload file and create vector store
        file_path = './document.md'
        print(f"Uploading file from: {file_path}")
        file = await agents_client_async.files.upload_and_poll(file_path=file_path, purpose="assistants")
        print(f"Uploaded file, file ID: {file.id}")

        vector_store = await agents_client_async.vector_stores.create_and_poll(file_ids=[file.id], name="contoso_knowledge_base")
        print(f"Created vector store, vector store ID: {vector_store.id}")

        # 2. Create file search tool with uploaded resources
        file_search_tool = AzureOpenAIResponsesClient.get_file_search_tool(vector_store_ids=[vector_store.id])

        # 3. Create an agent with file search capabilities using the provider
        agent = client.as_agent(
            name=agent_name,
            instructions=agent_instructions,
            tools=[file_search_tool])

        # 4. Query the agent
        print(f"\n# User: '{query}'")
        response = await agent.run(query)
        print(f"# Agent: {response.text}")
        
    finally:
        # 5. Cleanup: Delete the vector store and file to prevent orphaned resources
        if vector_store:
            await agents_client_async.vector_stores.delete(vector_store.id)
            print(f"\nDeleted vector store: {vector_store.id}")
        if file:
            await agents_client_async.files.delete(file.id)
            print(f"Deleted file: {file.id}")

Uploading file from: ./document.md
Uploaded file, file ID: assistant-D4nv2PmzrTcg5aY9f1kQ4h
Created vector store, vector store ID: vs_tP6V6SiBhF4ULadzyOQsMVwY

# User: 'Can you explain Contoso's travel insurance coverage?'
# Agent: Contoso's travel insurance covers the following:

- Medical emergencies
- Trip cancellations
- Lost baggage.

Deleted vector store: vs_tP6V6SiBhF4ULadzyOQsMVwY
Deleted file: assistant-D4nv2PmzrTcg5aY9f1kQ4h
